<a href="https://colab.research.google.com/github/Lee-hosuk/game-agent/blob/main/connect-four/connect_four.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔴🟡 사목 (Connect Four) - 알파-베타 탐색 에이전트

7칸 × 6줄 판에 위에서 돌을 떨어뜨려, **가로·세로·대각선으로 4개**를 먼저 잇는 쪽이 이깁니다.

## 틱택토와 뭐가 다를까?
틱택토의 미니맥스는 **게임 끝까지 모든 경우를 계산**했습니다. 경우의 수가 약 25만 개라 가능했죠.
사목은 가능한 판 상태가 약 **4조 5천억 개**라서, 파이썬으로 끝까지 계산하는 건 불가능합니다.

그래서 이번에는 실제 게임 AI(체스, 바둑의 고전 AI)가 쓰는 세 가지 아이디어를 직접 구현합니다.

| 아이디어 | 설명 |
|---|---|
| **깊이 제한 탐색** | 끝까지 말고 **몇 수 앞까지만** 내다본다 |
| **평가 함수** | 게임이 안 끝난 판을 **점수로 어림잡는다** ("이 판은 내가 유리해 보임") |
| **알파-베타 가지치기** | 볼 필요 없는 수는 **계산을 건너뛴다** → 같은 결과를 훨씬 빨리 |

## 1. 사목 게임 만들기

In [ ]:
import random
import time
import math

ROWS, COLS = 6, 7
EMPTY = " "
PIECE_EMOJI = {"X": "🔴", "O": "🟡", EMPTY: "⚪"}

def new_board():
    return [[EMPTY] * COLS for _ in range(ROWS)]      # board[행][열], 0행이 맨 위

def legal_moves(board):
    return [c for c in range(COLS) if board[0][c] == EMPTY]   # 맨 윗칸이 비어 있는 열

def drop(board, col, piece):
    """col 열에 돌을 떨어뜨림. 새 판을 돌려줌 (원래 판은 그대로)"""
    new = [row[:] for row in board]
    for r in range(ROWS - 1, -1, -1):          # 아래에서부터 빈칸 찾기
        if new[r][col] == EMPTY:
            new[r][col] = piece
            return new
    raise ValueError("꽉 찬 열")

def winner(board):
    for r in range(ROWS):
        for c in range(COLS):
            p = board[r][c]
            if p == EMPTY:
                continue
            for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:   # 가로, 세로, 대각선 두 방향
                if all(0 <= r + dr*i < ROWS and 0 <= c + dc*i < COLS
                       and board[r + dr*i][c + dc*i] == p for i in range(4)):
                    return p
    if not legal_moves(board):
        return "draw"
    return None

def show(board):
    print(" ".join(f"{c+1}\ufe0f\u20e3" for c in range(COLS)))
    for row in board:
        print(" ".join(PIECE_EMOJI[p] for p in row))

b = new_board()
for col, piece in [(3, "X"), (3, "O"), (2, "X")]:
    b = drop(b, col, piece)
show(b)
print("둘 수 있는 열:", [c + 1 for c in legal_moves(b)])

## 2. 대전 함수와 랜덤 봇
플레이어는 `(판, 내 돌) → 열 번호(0~6)`를 돌려주는 함수입니다. 🔴(X)가 먼저 둡니다.

In [ ]:
def random_bot(board, me):
    return random.choice(legal_moves(board))

def play_game(player_x, player_o, verbose=False, record=None):
    board = new_board()
    players = {"X": player_x, "O": player_o}
    turn = "X"
    while winner(board) is None:
        col = players[turn](board, turn)
        if col not in legal_moves(board):          # 안전장치
            col = random.choice(legal_moves(board))
        board = drop(board, col, turn)
        if record is not None:
            record.append([row[:] for row in board])
        if verbose:
            print(f"{PIECE_EMOJI[turn]} → {col+1}번 열")
            show(board)
            print()
        turn = "O" if turn == "X" else "X"
    return winner(board)

random.seed(0)
print("결과:", play_game(random_bot, random_bot))

## 3. 평가 함수: 게임이 안 끝난 판에 점수 매기기 ⭐

몇 수 앞까지만 보면 대부분 게임이 끝나지 않은 상태에서 멈춥니다.
그때 "이 판이 나에게 얼마나 유리한가"를 **어림잡는 함수**가 필요합니다.

판에서 **연속된 4칸짜리 구간(window)**을 전부 살펴보고 점수를 줍니다. (가로·세로·대각선 합쳐서 69개)

| 4칸 구간 상태 | 점수 |
|---|---|
| 내 돌 3개 + 빈칸 1개 | **+5** (곧 이길 수 있음) |
| 내 돌 2개 + 빈칸 2개 | **+2** |
| 상대 돌 3개 + 빈칸 1개 | **−4** (막아야 함) |
| 가운데 열에 있는 내 돌 1개마다 | **+3** (가운데는 연결 가능한 줄이 가장 많음) |

In [ ]:
def all_windows(board):
    """판의 모든 4칸 구간 (가로, 세로, 대각선 두 방향)"""
    for r in range(ROWS):
        for c in range(COLS):
            for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                cells = [(r + dr*i, c + dc*i) for i in range(4)]
                if all(0 <= rr < ROWS and 0 <= cc < COLS for rr, cc in cells):
                    yield [board[rr][cc] for rr, cc in cells]

def score_window(window, me):
    opp = "O" if me == "X" else "X"
    mine, theirs, empty = window.count(me), window.count(opp), window.count(EMPTY)
    if mine == 3 and empty == 1:   return 5
    if mine == 2 and empty == 2:   return 2
    if theirs == 3 and empty == 1: return -4
    return 0

def evaluate(board, me):
    center = [board[r][COLS // 2] for r in range(ROWS)]
    score = center.count(me) * 3
    for w in all_windows(board):
        score += score_window(w, me)
    return score

print("구간 개수:", len(list(all_windows(new_board()))))
print("위 예시 판 점수 (🔴 입장):", evaluate(b, "X"))
print("위 예시 판 점수 (🟡 입장):", evaluate(b, "O"))

## 4. 깊이 제한 미니맥스

틱택토의 미니맥스와 같지만, **정해진 깊이까지만** 내려가고 거기서 평가 함수로 점수를 매깁니다.
- 내 차례: 점수가 **가장 높은** 수를 고름 (max)
- 상대 차례: 상대는 나에게 **가장 불리한** 수를 고른다고 가정 (min)

이기면 아주 큰 점수를 주는데, **남은 깊이만큼 더 줘서** 같은 승리라도 빨리 이기는 쪽을 고르게 합니다.

`nodes`로 판을 몇 개나 살펴봤는지 셉니다. 다음 셀의 알파-베타와 비교할 거예요.

In [ ]:
WIN_SCORE = 1_000_000
nodes = 0     # 살펴본 판의 개수

def terminal_score(board, me, depth):
    w = winner(board)
    if w == me:     return WIN_SCORE + depth     # 빨리 이길수록 높은 점수
    if w == "draw": return 0
    if w is not None: return -WIN_SCORE - depth  # 늦게 질수록 덜 나쁨
    return None

def minimax(board, depth, maximizing, me):
    global nodes
    nodes += 1
    t = terminal_score(board, me, depth)
    if t is not None:
        return t
    if depth == 0:
        return evaluate(board, me)
    opp = "O" if me == "X" else "X"
    if maximizing:
        return max(minimax(drop(board, c, me), depth - 1, False, me) for c in legal_moves(board))
    else:
        return min(minimax(drop(board, c, opp), depth - 1, True, me) for c in legal_moves(board))

## 5. 알파-베타 가지치기 ✂️

미니맥스는 모든 수를 다 봅니다. 그런데 **어차피 선택되지 않을 수**는 끝까지 볼 필요가 없습니다.

> 내가 A수를 두면 최소 10점은 보장된다는 걸 알았다.
> B수를 살펴보는데, 상대가 B에 대한 응수 하나로 3점까지 끌어내릴 수 있다는 게 보였다.
> → B의 나머지 응수는 볼 필요가 없다. B는 아무리 좋아도 3점 이하라서 A(10점)보다 못하니까.

- **alpha**: 내가 지금까지 확보한 최고 점수 (이보다 나쁜 건 안 고름)
- **beta**: 상대가 지금까지 허용한 최저 점수 (상대는 이보다 좋은 걸 안 줌)
- **alpha ≥ beta**가 되면 → 나머지 수는 **잘라냄**

결과는 미니맥스와 **완전히 똑같고**, 살펴보는 판의 개수만 크게 줄어듭니다.
가운데 열부터 살펴보면 좋은 수를 일찍 찾아서 더 많이 잘라낼 수 있습니다.

In [ ]:
ORDER = [3, 2, 4, 1, 5, 0, 6]      # 가운데부터 살펴보기

def alphabeta(board, depth, alpha, beta, maximizing, me):
    global nodes
    nodes += 1
    t = terminal_score(board, me, depth)
    if t is not None:
        return t
    if depth == 0:
        return evaluate(board, me)
    opp = "O" if me == "X" else "X"
    moves = [c for c in ORDER if c in legal_moves(board)]
    if maximizing:
        value = -math.inf
        for c in moves:
            value = max(value, alphabeta(drop(board, c, me), depth - 1, alpha, beta, False, me))
            alpha = max(alpha, value)
            if alpha >= beta:
                break                     # ✂️ 가지치기: 상대가 이 쪽으로 오게 두지 않음
        return value
    else:
        value = math.inf
        for c in moves:
            value = min(value, alphabeta(drop(board, c, opp), depth - 1, alpha, beta, True, me))
            beta = min(beta, value)
            if alpha >= beta:
                break                     # ✂️ 가지치기: 나는 이 쪽을 고르지 않음
        return value

### 미니맥스 vs 알파-베타 비교
같은 판, 같은 깊이에서 **고른 수와 점수는 같은데 살펴본 판 개수는 얼마나 다른지** 확인합니다.

In [ ]:
def best_move(board, me, depth, search):
    """search: 'minimax' 또는 'alphabeta'. 각 수의 점수를 계산해 가장 좋은 수를 고름"""
    scores = {}
    for c in legal_moves(board):
        child = drop(board, c, me)
        if search == "minimax":
            scores[c] = minimax(child, depth - 1, False, me)
        else:
            scores[c] = alphabeta(child, depth - 1, -math.inf, math.inf, False, me)
    best = max(scores.values())
    return random.choice([c for c, s in scores.items() if s == best]), scores

test = new_board()
for col, piece in [(3, "X"), (3, "O"), (2, "X"), (4, "O")]:
    test = drop(test, col, piece)
show(test)
print()
print(f"{'깊이':>4} | {'미니맥스 판 수':>12} | {'알파-베타 판 수':>13} | {'줄어든 비율':>8} | 점수 같음?")
for depth in [2, 3, 4, 5]:
    nodes = 0; t0 = time.time()
    _, s1 = best_move(test, "X", depth, "minimax")
    n1, t1 = nodes, time.time() - t0
    nodes = 0; t0 = time.time()
    _, s2 = best_move(test, "X", depth, "alphabeta")
    n2, t2 = nodes, time.time() - t0
    print(f"{depth:>4} | {n1:>12,} | {n2:>13,} | {100*(1-n2/n1):>7.0f}% | {s1 == s2}  ({t1:.2f}초 → {t2:.2f}초)")

**볼 포인트**
- 깊이가 1 늘 때마다 미니맥스의 판 수는 약 **7배씩** 늘어납니다. (한 수마다 선택지가 7개)
- 알파-베타는 **같은 점수**를 내면서 훨씬 적은 판만 봅니다. 그래서 같은 시간에 **더 깊이** 볼 수 있습니다.

## 6. 알파-베타 에이전트 만들기
깊이를 정해서 플레이어 함수를 만듭니다. 깊이가 깊을수록 강하지만 느립니다.

In [ ]:
def make_ab_player(depth):
    def ab_player(board, me):
        col, _ = best_move(board, me, depth, "alphabeta")
        return col
    ab_player.__name__ = f"alphabeta_d{depth}"
    return ab_player

ab1, ab2, ab4 = make_ab_player(1), make_ab_player(2), make_ab_player(4)

random.seed(1)
record = []
result = play_game(ab4, random_bot, record=record)
show(record[-1])
print("결과:", {"X": "🔴 알파-베타(깊이 4) 승", "O": "🟡 랜덤 승", "draw": "무승부"}[result], f"/ {len(record)}수")

## 7. 리그전: 깊이에 따라 얼마나 강해질까? 🏆
선공/후공을 번갈아 가며 여러 판 둡니다. (사목은 선공이 유리한 게임이라 공평하게 바꿔줘야 해요)

⏱️ 코랩 CPU 기준 1분 안팎 걸립니다. 같은 깊이끼리는 거의 같은 수를 두기 때문에, 판 수가 적으면 운(선공 유리함)에 크게 좌우됩니다. 그래서 12판씩 둡니다.

In [ ]:
def match(p1, p2, n_games=6, label1="", label2=""):
    rec = {"win": 0, "draw": 0, "loss": 0}
    for g in range(n_games):
        if g % 2 == 0:
            r = play_game(p1, p2); mark = "X"
        else:
            r = play_game(p2, p1); mark = "O"
        rec["draw" if r == "draw" else ("win" if r == mark else "loss")] += 1
    print(f"{label1:<14} vs {label2:<14} | 승 {rec['win']} / 무 {rec['draw']} / 패 {rec['loss']}")
    return rec

random.seed(0)
t0 = time.time()
league = {
    ("depth 1", "random"):  match(ab1, random_bot, 10, "깊이 1", "랜덤"),
    ("depth 2", "depth 1"): match(ab2, ab1, 12, "깊이 2", "깊이 1"),
    ("depth 4", "depth 2"): match(ab4, ab2, 12, "깊이 4", "깊이 2"),
}
print(f"\n걸린 시간: {time.time() - t0:.0f}초")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = [f"{a}\nvs {b}" for a, b in league]
win  = np.array([r["win"]  for r in league.values()])
draw = np.array([r["draw"] for r in league.values()])
loss = np.array([r["loss"] for r in league.values()])
total = win + draw + loss

plt.figure(figsize=(7, 4))
plt.bar(labels, win / total * 100, label="win", color="#4caf50")
plt.bar(labels, draw / total * 100, bottom=win / total * 100, label="draw", color="#9e9e9e")
plt.bar(labels, loss / total * 100, bottom=(win + draw) / total * 100, label="loss", color="#f44336")
plt.ylabel("%")
plt.title("Connect Four: deeper search wins more")
plt.legend()
plt.savefig("c4_league.png", dpi=100, bbox_inches="tight")
plt.show()

## 8. 경기 GIF 만들기 🎞️
알파-베타(깊이 4)끼리 둔 경기를 GIF로 저장합니다.

In [ ]:
import imageio
from IPython.display import Image, display
from matplotlib.patches import Circle

def board_image(board, title=""):
    fig, ax = plt.subplots(figsize=(3.5, 3.2), dpi=80)
    ax.set_xlim(0, COLS); ax.set_ylim(0, ROWS); ax.set_aspect("equal"); ax.axis("off")
    ax.add_patch(plt.Rectangle((0, 0), COLS, ROWS, color="#1e5bb8"))
    colors = {"X": "#e53935", "O": "#fdd835", EMPTY: "white"}
    for r in range(ROWS):
        for c in range(COLS):
            ax.add_patch(Circle((c + 0.5, ROWS - r - 0.5), 0.4, color=colors[board[r][c]]))
    ax.set_title(title, fontsize=10)
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
    plt.close(fig)
    return img

def save_game_gif(player_x, player_o, filename, title_x="", title_o=""):
    record = [new_board()]
    result = play_game(player_x, player_o, record=record)
    names = {"X": f"Red ({title_x})", "O": f"Yellow ({title_o})", "draw": "Draw"}
    frames = [board_image(b, f"move {i}") for i, b in enumerate(record)]
    frames.append(board_image(record[-1], f"{names[result]} wins!" if result != "draw" else "Draw"))
    durations = [500] * (len(frames) - 1) + [3000]     # 밀리초 단위. 마지막 장면은 3초 동안 보여줌
    imageio.mimsave(filename, frames, duration=durations, loop=0)
    return result

random.seed(3)
save_game_gif(ab4, ab2, "c4_game.gif", "depth 4", "depth 2")
display(Image(open("c4_game.gif", "rb").read()))

## 9. LLM도 사목을 둘 수 있을까? 🤖

틱택토 때처럼 `google.colab.ai`로 LLM을 붙여봅니다. (API 키 필요 없음)
판이 커서 틱택토보다 훨씬 어려운데, 알파-베타 에이전트와 비교하면 어떨까요?

In [ ]:
from google.colab import ai
import re

MODEL = None   # None이면 기본 모델. ai.list_models()에서 골라 넣어도 됨

def ask_llm(prompt):
    return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

def board_to_text(board):
    lines = [" ".join(str(c + 1) for c in range(COLS))]
    lines += [" ".join(p if p != EMPTY else "." for p in row) for row in board]
    return "\n".join(lines)

LLM_PROMPT = """너는 사목(Connect Four)을 두고 있어. 너는 '{me}'이고 상대는 '{opp}'야.
7열 6줄 판이고, 열 번호를 고르면 돌이 그 열의 가장 아래 빈칸으로 떨어져.
가로, 세로, 대각선으로 같은 돌 4개를 먼저 이으면 이겨. '.'은 빈칸이야.

현재 판 (맨 윗줄은 열 번호, 아래로 갈수록 바닥):
{board}

둘 수 있는 열: {legal}

1. 내가 지금 두면 바로 4개가 이어지는 열이 있나?
2. 상대가 다음에 두면 4개가 이어지는 열이 있나? 있으면 막아.
3. 없으면 가운데 쪽, 그리고 3개를 만들 수 있는 열을 골라.
짧게 이유를 말하고, 마지막 줄에 반드시 'MOVE: 열번호' 형식으로 답해."""

llm_stats = {"calls": 0, "invalid": 0}

def make_llm_player(show_reason=True, max_tries=3):
    def llm_player(board, me):
        opp = "O" if me == "X" else "X"
        legal = [c + 1 for c in legal_moves(board)]
        prompt = LLM_PROMPT.format(me=me, opp=opp, board=board_to_text(board), legal=legal)
        for _ in range(max_tries):
            llm_stats["calls"] += 1
            try:
                answer = ask_llm(prompt)
            except Exception as e:
                print("LLM 오류:", e); time.sleep(5); continue
            m = re.findall(r"MOVE\s*[:：]\s*([1-7])", answer, re.IGNORECASE)
            col = int(m[-1]) if m else None
            if col in legal:
                if show_reason:
                    print(f"🤖 LLM({PIECE_EMOJI[me]}) 생각: {answer.strip()}\n")
                return col - 1
            llm_stats["invalid"] += 1
            prompt += f"\n\n{col}번 열은 둘 수 없어. 다음 중에서 골라: {legal}"
        return random.choice(legal_moves(board))
    return llm_player

print(board_to_text(test))

LLM vs 알파-베타(깊이 2) 한 판. LLM이 🔴(선공)입니다. ⏱️ LLM을 20번 안팎 호출해서 1~2분 걸릴 수 있어요.

In [ ]:
random.seed(0)
llm_stats.update(calls=0, invalid=0)
record = []
result = play_game(make_llm_player(show_reason=False), ab2, record=record)
show(record[-1])
print("결과:", {"X": "🔴 LLM 승", "O": "🟡 알파-베타(깊이 2) 승", "draw": "무승부"}[result])
print("LLM 호출:", llm_stats["calls"], "| 잘못된 수:", llm_stats["invalid"])

## 10. 직접 대결하기 🎮
열 버튼을 눌러 돌을 떨어뜨리세요. 당신이 🔴(선공)입니다.
- `opponent`를 바꾸면 상대를 고를 수 있어요: `make_ab_player(1~5)` 또는 `make_llm_player()`
- 깊이 5 이상은 한 수에 몇 초씩 걸릴 수 있어요.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def play_vs_ai(opponent, human="X"):
    ai_mark = "O" if human == "X" else "X"
    state = {"board": new_board()}
    col_buttons = [widgets.Button(description=f"↓ {c+1}", layout=widgets.Layout(width="48px")) for c in range(COLS)]
    board_view = widgets.HTML()
    status = widgets.HTML()
    log = widgets.Output()

    def render():
        b = state["board"]
        rows = "".join("<div>" + "".join(PIECE_EMOJI[p] for p in row) + "</div>" for row in b)
        board_view.value = f"<div style='font-size:34px; line-height:40px; letter-spacing:4px'>{rows}</div>"
        over = winner(b) is not None
        for c, btn in enumerate(col_buttons):
            btn.disabled = over or c not in legal_moves(b)

    def check_end():
        w = winner(state["board"])
        if w is None:
            return False
        status.value = {"draw": "<h3>🤝 무승부!</h3>", human: "<h3>🎉 당신이 이겼어요!</h3>"}.get(w, "<h3>🤖 AI가 이겼어요!</h3>")
        render()
        return True

    def ai_turn():
        status.value = "<h3>🤖 AI가 생각 중...</h3>"
        for btn in col_buttons:
            btn.disabled = True
        with log:
            clear_output()
            col = opponent(state["board"], ai_mark)
        state["board"] = drop(state["board"], col, ai_mark)
        render()
        if not check_end():
            status.value = f"<h3>당신({PIECE_EMOJI[human]}) 차례예요. 열을 고르세요. (AI는 {col+1}번 열에 뒀어요)</h3>"

    def on_click(btn):
        c = col_buttons.index(btn)
        if winner(state["board"]) is not None or c not in legal_moves(state["board"]):
            return
        state["board"] = drop(state["board"], c, human)
        render()
        if not check_end():
            ai_turn()

    for btn in col_buttons:
        btn.on_click(on_click)
    display(widgets.VBox([status, widgets.HBox(col_buttons), board_view, log]))
    render()
    if human == "O":
        ai_turn()
    else:
        status.value = f"<h3>당신({PIECE_EMOJI[human]}) 차례예요. 열을 고르세요.</h3>"

opponent = make_ab_player(4)      # 상대 바꾸기: make_ab_player(2), make_llm_player() 등
play_vs_ai(opponent, human="X")   # 후공으로 하려면 human="O"

## 🔥 도전 과제
1. **평가 함수 바꾸기**: `score_window`의 점수(5, 2, −4)나 가운데 보너스(3)를 바꾸면 강해질까요? 바꾼 버전끼리 `match`로 붙여보세요.
2. **가지치기 순서 실험**: `ORDER`를 `[0, 1, 2, 3, 4, 5, 6]`으로 바꾸면 5번 셀의 알파-베타 판 수가 어떻게 변할까요?
3. **시간 제한 탐색**: 깊이를 고정하지 말고 "1초 안에서 최대한 깊이" 보도록 바꿔보세요. (반복 심화, iterative deepening)
4. **LLM 리그**: LLM을 랜덤, 깊이 1, 깊이 2와 각각 붙여서 LLM이 어느 정도 실력인지 재보세요.
5. **강화학습과 비교**: 사목은 Q-테이블로는 너무 큽니다. 왜 그런지, 그리고 DQN이나 알파제로 방식이 왜 필요한지 생각해보세요.